# Pediatric Brain Tumor Segmentation (nnU-Net v2 baseline)

Publication-ready notebook: minimal, reproducible pipeline to (1) locate BraTS-PEDs data, (2) export to nnU-Net v2 raw format, (3) run planning/preprocessing, (4) run 5-fold CV training, and (5) monitor progress from logs/checkpoints.

Note: training is designed to be **disconnect-safe** (background `nohup` runner + log files).

## Citations

If you use nnU-Net, please cite:
- Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). *nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation*. **Nature Methods, 18**(2), 203–211.

In [ ]:
# Section 0 — Reproducibility
from __future__ import annotations

import os
import platform
import random
from dataclasses import dataclass
from pathlib import Path

import numpy as np

DEFAULT_SEED = 1337


def seed_everything(seed: int = DEFAULT_SEED, deterministic: bool = True) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch

        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        if deterministic:
            torch.backends.cudnn.deterministic = True
            torch.backends.cudnn.benchmark = False
            torch.use_deterministic_algorithms(False)
    except Exception as exc:
        print(f"[WARN] Torch seeding skipped: {exc}")


@dataclass(frozen=True)
class Paths:
    extraction_base: Path
    training_main_dir: Path
    validation_main_dir: Path


seed_everything(DEFAULT_SEED)
print("Seeded.")
print("Platform:", platform.platform())
print("Python:", platform.python_version())
print("CWD:", Path.cwd())

In [ ]:
# Section 1A — Environment sanity checks
import importlib
import shutil
import sys


def _try_import(pkg: str) -> bool:
    try:
        mod = importlib.import_module(pkg)
        ver = getattr(mod, "__version__", "unknown")
        print(f"[OK] import {pkg} (version={ver})")
        return True
    except Exception as exc:
        print(f"[MISSING] import {pkg} failed: {exc}")
        return False


print("Executable:", sys.executable)
_try_import("numpy")
_try_import("nibabel")
_try_import("SimpleITK")
has_torch = _try_import("torch")
if has_torch:
    import torch

    print("torch:", torch.__version__)
    print("cuda available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("gpu:", torch.cuda.get_device_name(0))

print()
for cmd in [
    "nnUNetv2_plan_and_preprocess",
    "nnUNetv2_train",
    "nnUNetv2_predict",
]: 
    p = shutil.which(cmd)
    print(f"{cmd}: {'FOUND' if p else 'NOT FOUND'}" + (f" -> {p}" if p else ""))

In [ ]:
# Section 1B — Locate BraTS-PEDs dataset root
from pathlib import Path


def _find_brats_peds_root(preferred: Path) -> Path | None:
    preferred = Path(preferred)
    if preferred.exists():
        return preferred

    # Fallback (bounded): look under a few common locations
    candidates = [Path.cwd(), Path.cwd() / 'archive', Path.cwd().parent, Path.cwd().parent / 'archive']
    target_names = {"BraTS-PEDs2024_Training", "BraTS-PEDs2024_Validation"}

    for base in candidates:
        if not base.exists():
            continue
        try:
            base_parts = len(base.resolve().parts)
            for p in base.rglob('*'):
                try:
                    if len(p.resolve().parts) - base_parts > 4:
                        continue
                except Exception:
                    continue
                if p.is_dir() and p.name in target_names:
                    return p.parent.parent
        except Exception:
            continue
    return None


def _find_validation_dir(extraction_base: Path) -> Path | None:
    extraction_base = Path(extraction_base)
    expected = extraction_base / 'validation' / 'BraTS-PEDs2024_Validation'
    if expected.exists():
        return expected
    variants = [
        extraction_base / 'validation' / 'BraTS-PEDs2024_ValidationData',
        extraction_base / 'BraTS-PEDs2024_Validation',
        extraction_base / 'BraTS-PEDs2024_ValidationData',
    ]
    for v in variants:
        if v.exists():
            return v
    try:
        for p in extraction_base.rglob('BraTS-PEDs2024_*Validation*'):
            if p.is_dir():
                return p
    except Exception:
        pass
    return None


preferred_root = Path('/workspace/pediatric_tumor_data')
extraction_base = _find_brats_peds_root(preferred_root)
if extraction_base is None:
    raise FileNotFoundError(
        "Could not find PEDs dataset root. Expected GPU mount '/workspace/pediatric_tumor_data'. \n"
        "If running locally, set preferred_root appropriately."
    )

training_main_dir = extraction_base / 'training' / 'BraTS-PEDs2024_Training'
validation_main_dir = _find_validation_dir(extraction_base)
paths = Paths(
    extraction_base=extraction_base,
    training_main_dir=training_main_dir,
    validation_main_dir=(validation_main_dir if validation_main_dir is not None else Path('')),
)

print('extraction_base:', extraction_base)
print('training_main_dir:', training_main_dir, 'exists=', training_main_dir.exists())
print('validation_main_dir:', validation_main_dir if validation_main_dir is not None else '<not found>')
if not training_main_dir.exists():
    raise FileNotFoundError('Training folder missing at training/BraTS-PEDs2024_Training')

train_cases = sorted([p for p in training_main_dir.iterdir() if p.is_dir()])
val_cases = (
    sorted([p for p in validation_main_dir.iterdir() if p.is_dir()])
    if validation_main_dir is not None and validation_main_dir.exists()
    else []
)
print(f'#train cases: {len(train_cases)}')
print(f'#val cases (folder): {len(val_cases)}')
print('Example train case:', train_cases[0] if train_cases else '<none>')

In [ ]:
# Section 1C — Label mapping (source -> BraTS-style {0,1,2,4})
#
# If your source labels already are {0,1,2,4}, keep identity.
# If your source contains label 3, you must decide how to map it before export.
# Default below maps 3 -> 1 (tumor core / non-enhancing tumor). Adjust if needed.

from pathlib import Path
import numpy as np
import nibabel as nib

LABEL_MAPPING = {0: 0, 1: 1, 2: 2, 3: 1, 4: 4}

def _index_case_files(case_dir: Path):
    files = list(case_dir.glob('*.nii')) + list(case_dir.glob('*.nii.gz'))
    seg = None
    for f in files:
        name = f.name.lower()
        if 'seg' in name and '.nii' in name:
            seg = f
            break
    return seg

def _sample_unique_labels(case_dirs: list[Path], n_cases: int = 20) -> set[int]:
    uniq: set[int] = set()
    seen = 0
    for d in case_dirs:
        seg = _index_case_files(d)
        if not seg:
            continue
        arr = np.asanyarray(nib.load(str(seg)).dataobj)
        uniq |= set(np.unique(arr).astype(int).tolist())
        seen += 1
        if seen >= n_cases:
            break
    return uniq

uniq_train = _sample_unique_labels(train_cases, n_cases=25)
print('Sampled unique labels (train):', sorted(uniq_train))
print('LABEL_MAPPING:', LABEL_MAPPING)
missing = set(uniq_train) - set(LABEL_MAPPING.keys())
if missing:
    raise RuntimeError(f'Mapping missing keys for observed labels: {sorted(missing)}')
print('[OK] Mapping covers observed labels in sample.')

In [ ]:
# Section 2 — nnU-Net v2 setup (paths + dataset id)
import os
import shutil
from pathlib import Path

def _ensure_dir(p: Path) -> Path:
    p.mkdir(parents=True, exist_ok=True)
    return p

# Keep everything under extraction_base for locality
base_root = Path(str(training_main_dir)).parents[1]  # e.g. /workspace/pediatric_tumor_data
nnunet_base = _ensure_dir(base_root / 'nnunetv2')
nnUNet_raw = _ensure_dir(nnunet_base / 'nnUNet_raw')
nnUNet_preprocessed = _ensure_dir(nnunet_base / 'nnUNet_preprocessed')
nnUNet_results = _ensure_dir(nnunet_base / 'nnUNet_results')

os.environ['nnUNet_raw'] = str(nnUNet_raw)
os.environ['nnUNet_preprocessed'] = str(nnUNet_preprocessed)
os.environ['nnUNet_results'] = str(nnUNet_results)

DATASET_ID = 501
DATASET_NAME = 'BraTS_PEDs2024'
DATASET_FOLDER = nnUNet_raw / f'Dataset{DATASET_ID:03d}_{DATASET_NAME}'

print('nnUNet_raw:', nnUNet_raw)
print('nnUNet_preprocessed:', nnUNet_preprocessed)
print('nnUNet_results:', nnUNet_results)
print('DATASET_FOLDER:', DATASET_FOLDER)

for cmd in ['nnUNetv2_plan_and_preprocess', 'nnUNetv2_train', 'nnUNetv2_predict']:
    print(cmd + ':', shutil.which(cmd) or 'NOT FOUND on PATH')

In [ ]:
# Section 3 — Export BraTS-PEDs to nnU-Net v2 raw format
import json
import os
import re
import shutil
from pathlib import Path

import numpy as np
import nibabel as nib

OVERWRITE_DATASET = False

def _safe_mkdir(p: Path) -> None:
    p.mkdir(parents=True, exist_ok=True)

def _count_nii_gz(p: Path) -> int:
    return len(list(p.glob('*.nii.gz')))

def _link_or_copy(src: Path, dst: Path) -> None:
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists():
        return
    try:
        os.link(str(src), str(dst))
    except Exception:
        shutil.copy2(str(src), str(dst))

EXPORT_MODALITY_PATTERNS = {
    't1': [re.compile(r'(^|[^a-z0-9])t1n([^a-z0-9]|$)'), re.compile(r'(^|[^a-z0-9])t1([^a-z0-9]|$)')],
    't1ce': [re.compile(r'(^|[^a-z0-9])t1c([^a-z0-9]|$)'), re.compile(r't1ce|t1gd|t1c\b')],
    't2': [re.compile(r'(^|[^a-z0-9])t2w([^a-z0-9]|$)'), re.compile(r'(^|[^a-z0-9])t2([^a-z0-9]|$)')],
    'flair': [re.compile(r'(^|[^a-z0-9])t2f([^a-z0-9]|$)'), re.compile(r'flair')],
}

def _any_match(pats, text: str) -> bool:
    return any(p.search(text) for p in pats)

def _index_modalities_and_seg(case_dir: Path):
    files = list(case_dir.glob('*.nii')) + list(case_dir.glob('*.nii.gz'))
    if not files:
        raise FileNotFoundError(f'No NIfTI files in {case_dir}')
    mods = {}
    seg = None
    for f in files:
        name = f.name.lower()
        if 'seg' in name and '.nii' in name:
            seg = f
            continue
        for mod_key, pats in EXPORT_MODALITY_PATTERNS.items():
            if _any_match(pats, name):
                mods[mod_key] = f
                break
    return mods, seg

def _load_and_remap_seg(seg_path: Path, mapping: dict[int, int]) -> nib.Nifti1Image:
    img = nib.load(str(seg_path))
    data = img.get_fdata(dtype=np.float32)
    rounded = np.rint(data)
    max_abs_err = float(np.max(np.abs(data - rounded)))
    if max_abs_err > 1e-3:
        raise ValueError(f'Seg appears non-integer-coded: max_abs_err={max_abs_err} at {seg_path}')
    seg_int = rounded.astype(np.int16)

    out = np.zeros_like(seg_int, dtype=np.int16)
    for k, v in mapping.items():
        out[seg_int == int(k)] = int(v)

    uniq = set(np.unique(out).tolist())
    allowed = {0, 1, 2, 4}
    if not uniq.issubset(allowed):
        raise ValueError(f'Remapped seg contains unexpected labels {sorted(uniq)} from {seg_path}')
    return nib.Nifti1Image(out, img.affine, img.header)

imagesTr = DATASET_FOLDER / 'imagesTr'
labelsTr = DATASET_FOLDER / 'labelsTr'
imagesTs = DATASET_FOLDER / 'imagesTs'

expected_imagesTr = 4 * len(train_cases)
expected_labelsTr = len(train_cases)
expected_imagesTs = 4 * len(val_cases)

if DATASET_FOLDER.exists() and not OVERWRITE_DATASET:
    if (DATASET_FOLDER / 'dataset.json').exists() and imagesTr.exists() and labelsTr.exists() and imagesTs.exists():
        n_imgsTr = _count_nii_gz(imagesTr)
        n_lblsTr = _count_nii_gz(labelsTr)
        n_imgsTs = _count_nii_gz(imagesTs)
        if (n_imgsTr, n_lblsTr, n_imgsTs) == (expected_imagesTr, expected_labelsTr, expected_imagesTs):
            print('nnU-Net raw export already present; skipping rebuild.')
            print('imagesTr:', n_imgsTr, 'labelsTr:', n_lblsTr, 'imagesTs:', n_imgsTs)
        else:
            raise RuntimeError(
                'Dataset folder exists but counts do not match expected. '
                f'Found imagesTr={n_imgsTr} labelsTr={n_lblsTr} imagesTs={n_imgsTs}; '
                f'expected imagesTr={expected_imagesTr} labelsTr={expected_labelsTr} imagesTs={expected_imagesTs}. '
                'Set OVERWRITE_DATASET=True to rebuild.'
            )
    else:
        raise RuntimeError('Dataset folder exists but is missing expected subfolders/files. Set OVERWRITE_DATASET=True to rebuild.')

if OVERWRITE_DATASET:
    if DATASET_FOLDER.exists():
        shutil.rmtree(DATASET_FOLDER)
    for p in [imagesTr, labelsTr, imagesTs]:
        _safe_mkdir(p)

    channel_order = ['t1', 't1ce', 't2', 'flair']
    channel_names = {str(i): name for i, name in enumerate(['T1', 'T1ce', 'T2', 'FLAIR'])}
    label_names = {'background': 0, 'TC': 1, 'ED': 2, 'ET': 4}

    n_train_written = 0
    missing_train = 0
    for case_dir in train_cases:
        case_id = case_dir.name
        mods, seg = _index_modalities_and_seg(case_dir)
        missing = [m for m in channel_order if m not in mods]
        if missing or seg is None:
            missing_train += 1
            continue
        for i, mod_key in enumerate(channel_order):
            _link_or_copy(mods[mod_key], imagesTr / f'{case_id}_{i:04d}.nii.gz')
        remapped = _load_and_remap_seg(seg, LABEL_MAPPING)
        nib.save(remapped, str(labelsTr / f'{case_id}.nii.gz'))
        n_train_written += 1

    print(f'Training exported: {n_train_written} cases')
    print(f'Training skipped (missing data): {missing_train} cases')

    n_val_written = 0
    missing_val = 0
    for case_dir in val_cases:
        case_id = case_dir.name
        mods, _seg = _index_modalities_and_seg(case_dir)
        missing = [m for m in channel_order if m not in mods]
        if missing:
            missing_val += 1
            continue
        for i, mod_key in enumerate(channel_order):
            _link_or_copy(mods[mod_key], imagesTs / f'{case_id}_{i:04d}.nii.gz')
        n_val_written += 1

    print(f'imagesTs exported: {n_val_written} cases')
    print(f'imagesTs skipped (missing data): {missing_val} cases')

    dataset_json_path = DATASET_FOLDER / 'dataset.json'
    dataset_json = {
        'name': DATASET_NAME,
        'description': 'BraTS PEDs export with label remap to {0,1,2,4}',
        'tensorImageSize': '3D',
        'reference': 'BraTS PEDs',
        'licence': 'unknown',
        'release': '0.0',
        'channel_names': channel_names,
        'labels': {k: int(v) for k, v in label_names.items()},
        'numTraining': int(n_train_written),
        'file_ending': '.nii.gz',
    }
    dataset_json_path.write_text(json.dumps(dataset_json, indent=2))
    print('Wrote:', dataset_json_path)

In [ ]:
# Section 4 — nnU-Net label fix: make labels consecutive (ET 4 -> 3)
# nnU-Net expects consecutive label IDs {0,1,2,3,...}. BraTS uses ET=4, so convert 4->3.
import json
from pathlib import Path

import numpy as np
import nibabel as nib

labelsTr = DATASET_FOLDER / 'labelsTr'
dataset_json_path = DATASET_FOLDER / 'dataset.json'

def _unique_labels_in_file(p: Path) -> set[int]:
    arr = np.asanyarray(nib.load(str(p)).dataobj)
    return set(np.unique(arr).astype(int).tolist())

def _rewrite_et_4_to_3(p: Path) -> bool:
    img = nib.load(str(p))
    arr = np.asanyarray(img.dataobj).astype(np.int16)
    if 4 not in arr:
        return False
    arr = arr.copy()
    arr[arr == 4] = 3
    nib.save(nib.Nifti1Image(arr, img.affine, img.header), str(p))
    return True

lbl_files = sorted(labelsTr.glob('*.nii.gz'))
if not lbl_files:
    raise RuntimeError(f'No label files found in {labelsTr}')

sample_files = lbl_files[:: max(1, len(lbl_files)//10)]
uniqs = set()
for p in sample_files:
    uniqs |= _unique_labels_in_file(p)
print('Sampled unique labels (pre):', sorted(uniqs))

if 4 in uniqs:
    changed = 0
    for p in lbl_files:
        if _rewrite_et_4_to_3(p):
            changed += 1
    print('Converted ET 4->3 in label files touched:', changed)
else:
    print('No 4 detected in sample; assuming labels are already consecutive.')

uniqs_post = set()
for p in sample_files:
    uniqs_post |= _unique_labels_in_file(p)
print('Sampled unique labels (post):', sorted(uniqs_post))

if dataset_json_path.exists():
    ds = json.loads(dataset_json_path.read_text())
    if 'labels' in ds and 'ET' in ds['labels']:
        ds['labels']['ET'] = 3
        dataset_json_path.write_text(json.dumps(ds, indent=2))
        print('Updated dataset.json: ET=3')

NNUNET_LABEL_MAP = {0: 0, 1: 1, 2: 2, 3: 4}  # nnU-Net -> BraTS
print('NNUNET_LABEL_MAP (nnU-Net -> BraTS):', NNUNET_LABEL_MAP)

In [ ]:
# Section 5 — Fix .nii.gz files that are not actually gzip (SimpleITK compatibility)
import gzip
import os
import shutil
from pathlib import Path

imagesTr = DATASET_FOLDER / 'imagesTr'
imagesTs = DATASET_FOLDER / 'imagesTs'

def is_gzip_file(path: Path) -> bool:
    try:
        with gzip.open(path, 'rb') as f:
            f.read(1)
        return True
    except Exception:
        return False

def rewrite_as_gzip(path: Path) -> None:
    tmp = path.with_suffix(path.suffix + '.tmp')
    with open(path, 'rb') as fin, gzip.open(tmp, 'wb') as fout:
        shutil.copyfileobj(fin, fout)
    os.replace(str(tmp), str(path))

def fix_folder(folder: Path) -> int:
    fixed = 0
    for p in sorted(folder.glob('*.nii.gz')):
        if not is_gzip_file(p):
            rewrite_as_gzip(p)
            fixed += 1
    return fixed

fixed_tr = fix_folder(imagesTr)
fixed_ts = fix_folder(imagesTs)
print('Fixed mislabeled .nii.gz (not gzip):')
print(' - imagesTr:', fixed_tr)
print(' - imagesTs:', fixed_ts)

In [ ]:
# Section 6 — nnU-Net planning + preprocessing
# This is compute-heavy. Flip RUN=True when ready.
import os
import subprocess

RUN = False

def _cmd_help_contains(cmd: str, needle: str) -> bool:
    try:
        p = subprocess.run([cmd, '-h'], capture_output=True, text=True)
        txt = (p.stdout or '') + '\n' + (p.stderr or '')
        return needle in txt
    except Exception:
        return False

def run_cmd(cmd: str) -> None:
    print('$', cmd)
    p = subprocess.run(cmd, shell=True)
    if p.returncode != 0:
        raise RuntimeError(f'Command failed ({p.returncode}): {cmd}')

os.environ.setdefault('nnUNet_n_proc_DA', '8')
use_c_flag = _cmd_help_contains('nnUNetv2_plan_and_preprocess', '-c')
base = f'nnUNetv2_plan_and_preprocess -d {int(DATASET_ID)} --verify_dataset_integrity'
cmd = base + (' -c 3d_fullres' if use_c_flag else '')
print('Will run:')
print(cmd)
if RUN:
    run_cmd(cmd)
    print('\n[OK] Planning/preprocessing completed.')

In [ ]:
# Section 7 — Train folds 0–4 sequentially (disconnect-safe)
#
# Writes a bash script under: $nnUNet_results/notebook_runs
# Launches it via: nohup bash <script> > <log> 2>&1 &
#
# Safety: skips folds that already have checkpoint_final.pth

import os
import re
import shutil
import subprocess
import sys
import time
from pathlib import Path

CONFIG = '3d_fullres'
FOLDS = [0, 1, 2, 3, 4]
DA_WORKERS = 8
NNUNET_COMPILE = None  # set to 0 or 1 to force; None = do not override

CONFIRM_LAUNCH = False  # set True to actually launch

if os.name != 'posix':
    raise RuntimeError('This cell expects a Linux/posix runtime (uses bash/nohup/ps).')

results_root = Path(os.environ['nnUNet_results'])
run_root = results_root / 'notebook_runs'
run_root.mkdir(parents=True, exist_ok=True)

dataset_dir = results_root / f'Dataset{int(DATASET_ID):03d}_{DATASET_NAME}'
trainer_dir = dataset_dir / f'nnUNetTrainer__nnUNetPlans__{CONFIG}'
if not trainer_dir.exists():
    raise FileNotFoundError(f'Trainer dir not found: {trainer_dir} (did you run planning/preprocessing?)')

def _help_text(cmd: list[str]) -> str:
    p = subprocess.run(cmd, capture_output=True, text=True)
    return (p.stdout or '') + '\n' + (p.stderr or '')

help_txt = _help_text(['nnUNetv2_train', '-h'])
if re.search(r'\n\s*--c(\s|,|$)', help_txt):
    RESUME_FLAG = '--c'
elif re.search(r'\n\s*-c(\s|,|$)', help_txt):
    RESUME_FLAG = '-c'
elif '--continue' in help_txt:
    RESUME_FLAG = '--continue'
else:
    RESUME_FLAG = ''

print('dataset_dir:', dataset_dir)
print('trainer_dir:', trainer_dir)
print('resume flag detected:', RESUME_FLAG or '(none)')

# Do not overlap with any running training
ps = subprocess.run('ps -eo pid=,stat=,cmd=', shell=True, capture_output=True, text=True)
active = []
for ln in (ps.stdout or '').splitlines():
    parts = ln.strip().split(None, 2)
    if len(parts) < 3:
        continue
    pid_s, stat, cmd = parts[0], parts[1], parts[2]
    if 'nnUNetv2_train' in cmd and 'Z' not in stat and '<defunct>' not in cmd:
        active.append((pid_s, stat, cmd))
if active:
    print('Active nnUNetv2_train detected; not launching another runner:')
    for pid_s, stat, cmd in active[:5]:
        print(f'  pid={pid_s} stat={stat} cmd={cmd}')
    raise SystemExit

ts = time.strftime('%Y%m%d_%H%M%S')
script_path = run_root / f'seq_train_d{int(DATASET_ID)}_{CONFIG}_folds0to4_{ts}.sh'
sched_log = run_root / f'seq_train_d{int(DATASET_ID)}_{CONFIG}_folds0to4_{ts}.log'

env_exports = ['export PYTHONUNBUFFERED=1', f'export nnUNet_n_proc_DA={int(DA_WORKERS)}']
if NNUNET_COMPILE is not None:
    env_exports.append(f'export nnUNet_compile={int(NNUNET_COMPILE)}')
env_exports.append(f'export nnUNet_raw="{os.environ["nnUNet_raw"]}"')
env_exports.append(f'export nnUNet_preprocessed="{os.environ["nnUNet_preprocessed"]}"')
env_exports.append(f'export nnUNet_results="{os.environ["nnUNet_results"]}"')

folds_s = ' '.join(str(f) for f in FOLDS)
resume_flag_s = RESUME_FLAG
script = f'''#!/usr/bin/env bash
set -euo pipefail

{os.linesep.join(env_exports)}

echo "[seq] started: $(date -Is)"
echo "[seq] dataset={int(DATASET_ID)} config={CONFIG} folds={folds_s}"
echo "[seq] resume_flag={resume_flag_s}"

for f in {folds_s}; do
  FOLD_DIR="{trainer_dir}/fold_${{f}}"
  CK_FINAL="$FOLD_DIR/checkpoint_final.pth"
  CK_LATEST="$FOLD_DIR/checkpoint_latest.pth"
  CK_BEST="$FOLD_DIR/checkpoint_best.pth"

  if [[ -f "$CK_FINAL" ]]; then
    echo "[seq] fold $f: already complete, skipping"
    continue
  fi

  EXTRA=()
  if [[ -n "{resume_flag_s}" ]] && ([[ -f "$CK_LATEST" ]] || [[ -f "$CK_BEST" ]]); then
    EXTRA=("{resume_flag_s}")
    echo "[seq] fold $f: resuming"
  else
    echo "[seq] fold $f: starting fresh"
  fi

  OUT_LOG="{run_root}/nnunetv2_train_d{int(DATASET_ID)}_{CONFIG}_fold${{f}}_{ts}.log"
  echo "[seq] cmd: nnUNetv2_train {int(DATASET_ID)} {CONFIG} $f ${{EXTRA[*]}}" | tee -a "$OUT_LOG"
  nnUNetv2_train {int(DATASET_ID)} {CONFIG} "$f" ${{EXTRA[@]}} >> "$OUT_LOG" 2>&1

  if [[ -f "$CK_FINAL" ]]; then
    echo "[seq] fold $f: finished OK (checkpoint_final.pth exists)"
  else
    echo "[seq] fold $f: exited but checkpoint_final.pth missing -> stop"
    exit 2
  fi
done

echo "[seq] all done: $(date -Is)"
'''
script_path.write_text(script)
subprocess.run(['chmod', '+x', str(script_path)], check=False)

print('Wrote runner script:', script_path)
print('Runner log:', sched_log)
print('Per-fold logs will be written under:', run_root)

if not CONFIRM_LAUNCH:
    print('\nSet CONFIRM_LAUNCH=True to actually launch.')
else:
    cmd = f'nohup bash {script_path} > {sched_log} 2>&1 & echo $!'
    pid = subprocess.check_output(cmd, shell=True, text=True).strip()
    print('Launched runner PID:', pid)
    SEQ_RUN_PID = pid
    SEQ_RUN_SCHED_LOG = str(sched_log)
    SEQ_RUN_SCRIPT_PATH = str(script_path)

In [ ]:
# Section 8 — Progress dashboard (folds 0–4)
import os
import re
from datetime import datetime
from pathlib import Path

results_root = Path(os.environ['nnUNet_results'])
dataset_dir = results_root / f'Dataset{int(DATASET_ID):03d}_{DATASET_NAME}'
trainer_dir = dataset_dir / 'nnUNetTrainer__nnUNetPlans__3d_fullres'

def _find_latest_log(fold_dir: Path) -> Path | None:
    if not fold_dir.exists():
        return None
    logs = sorted(fold_dir.glob('training_log_*.txt'), key=lambda p: p.stat().st_mtime)
    return logs[-1] if logs else None

def _parse_last_epoch(log_path: Path) -> int | None:
    try:
        txt = log_path.read_text(errors='ignore')
    except Exception:
        return None
    tail = '\n'.join(txt.splitlines()[-300:])
    m = None
    for pat in [r'\bepoch\s*:\s*(\d+)\b', r'\bEpoch\s+(\d+)\b', r'\bepoch\s+(\d+)\b']:
        ms = list(re.finditer(pat, tail, flags=re.IGNORECASE))
        if ms:
            m = ms[-1]
            break
    return int(m.group(1)) if m else None

print('dataset_dir:', dataset_dir)
print('trainer_dir:', trainer_dir)
hdr = ['fold', 'status', 'last_epoch', 'last_update', 'log']
print('\t'.join(hdr))
for f in range(5):
    fold_dir = trainer_dir / f'fold_{f}'
    ck_final = fold_dir / 'checkpoint_final.pth'
    ck_latest = fold_dir / 'checkpoint_latest.pth'
    lp = _find_latest_log(fold_dir)
    last_epoch = _parse_last_epoch(lp) if lp else None
    if ck_final.exists():
        status = 'completed'
    elif ck_latest.exists():
        status = 'in-progress'
    elif fold_dir.exists():
        status = 'started?'
    else:
        status = 'not-started'
    cand = [p for p in [ck_latest if ck_latest.exists() else None, lp if lp and lp.exists() else None] if p]
    last_update = datetime.fromtimestamp(max(p.stat().st_mtime for p in cand)).isoformat(timespec='seconds') if cand else '-'
    print('\t'.join([str(f), status, str(last_epoch), last_update, str(lp) if lp else '-']))

## Optional — Inference on `imagesTs`

After training, predict on the exported `imagesTs` folder (and optionally ensemble folds 0–4). Example CLI (edit paths as needed):

- `nnUNetv2_predict -d 501 -i $nnUNet_raw/Dataset501_BraTS_PEDs2024/imagesTs -o <OUT_DIR> -c 3d_fullres -f 0 1 2 3 4`

If you need BraTS-style labels, map nnU-Net ET label `3` back to `4` using `NNUNET_LABEL_MAP` (defined in Section 4).